# 📚 LangChain ile Retrieval Augmented Generation'a Giriş 🦜🔗

Bu not defterinde LangChain kullanarak Retrieval Augmented Generation'ı nasıl kullanacağınızı öğreneceksiniz.

Kendi belgelerimiz hakkında sorular sormak için bir LLM kullanacağız!

## ⚙️ Kurulum

👉 Temel kütüphaneleri içe aktarmak için aşağıdaki hücreyi çalıştırın.

In [1]:
%load_ext autoreload
%autoreload 2
import os
from pprint import pprint
from IPython.display import Markdown

👉 API anahtarımızı tekrar yüklemek için aşağıdaki hücreyi çalıştırın:

In [2]:
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

## 📚 Neden RAG?

Bir LLM kendi başına, öğrendiği her şey hakkında sorulara yanıt verebilir.

Bunun birkaç dezavantajı vardır:
- Eğitim verileri geçmişten gelir ve en son verilerle güncellenmez.
- Sadece eğitim aldığı verileri bilir.

Bir LLM'yi kendi verilerimizle çalışması için kullanmak istiyoruz. İşte bu noktada RAG (Retrieval-Augmented Generation) devreye girer.

1. **Retrieval-Augmented Generation (RAG)**, gerçek doğruluğu artırmak için bir dil modelini belge alıcı ile birleştirir.
2. **İlgili dış belgeleri alır** (örneğin, bilgi tabanından) yanıtlar üretmeden önce.
3. **Dil modeli hem istemi hem de alınan bağlamı kullanarak** daha bilgili ve temelli çıktılar üretir.

## 🇪🇺 Bağlam

Bu meydan okumada, Avrupa Parlamentosu'ndan belgelerle çalışacağız.

Bir gazeteci olduğunuzu ve Avrupa Parlamentosu'nun genel kurul oturumları sırasında belirli bir konu hakkında neler söylendiğini öğrenmek istediğinizi düşünün. Bu oturumlar yılda 12 kez Strasbourg'da gerçekleşir ve 4 gün sürer. Oturumların transkriptleri EP'nin web sitesinde mevcuttur.

Kesinlikle tüm bu transkriptleri karıştırmak istemezsiniz. O halde, hayatımızı kolaylaştırmak için RAG'ı kullanalım!

Bu, her zaman test etmek için yepyeni veriler alabileceğimiz için çalışmak üzere iyi verilerdir.

## 📘 Verileri alalım

1. [EP'nin web sitesine](https://www.europarl.europa.eu/plenary/en/debates-video.html) gidin. 
1. Bu sizi en son genel kurul oturumuna yönlendirecektir.
1. İlk tarihin altında, "▶️ Verbatim reports HTML" bölümünde `HTML`'e tıklayın.
1. Sayfanın sonuna kaydırın ve alttaki PDF dosyasını indirin.
1. Dosyayı `data` klasörüne kaydedin.

Bir belgeyle başlayacağız, ancak daha sonrası için diğer birkaç günün aynısını şimdiden indirebilirsiniz.

Belgeye bir göz atın. Kaç sayfası var? Belge hakkında bir fikir edinmek için hızlıca belgede gezinin.

## 🔢 Belgeleri gömme

Belgeleri gömmek, tüm belgeleri veya belge parçalarını vektörlere çevirmek anlamına gelir.

LangChain🦜🔗 yine çok yardımcı olacak.

Bir gömme aracı (embedder) başlatalım ve deneyelim. LLM olarak Gemini kullandığımız için, Google'ın metin gömme araçlarında kalalım.

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

👉 Basit bir metin parçasını gömmek için gömme aracının `.embed_query()` metodunu deneyin.

In [4]:
# Embed a text like "What is the capital of France?" and save it to a variable `sample_embedding`

sample_embedding = embeddings.embed_query("What is the capital of France?")


👉 Bu `sample_embedding`'i keşfetmek için zaman ayırın. Nasıl görünüyor? Tipi nedir? Gömme boyutu nedir?

In [5]:
print(f"Tipi: {type(sample_embedding)}")
print(f"Gömme Boyutu (Uzunluk): {len(sample_embedding)}")
print(f"İlk 5 sayı: {sample_embedding[:5]}")

Tipi: <class 'list'>
Gömme Boyutu (Uzunluk): 3072
İlk 5 sayı: [-0.032554302, 0.01305496, 0.015067407, -0.07128718, -0.031062366]


## 💾 PDF'den gerçek verilerimizi yükle

Artık bir gömmenin nasıl göründüğünü biliyoruz, gerçek verilerimizle çalışmanın zamanı geldi.

👉 [LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/document_loaders/index#pdfs) gidin ve PyPDF kullanarak bir PDF'yi nasıl yükleyebileceğinizi öğrenin.

👉 Sonra devam edin ve daha önce indirdiğiniz PDF'lerden birini yükleyin.

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader

# 'data' klasöründeki PDF dosyasını otomatik buluyoruz
pdf_files = [f for f in os.listdir("data") if f.endswith(".pdf")]
pdf_path = f"data/{pdf_files[0]}"
print(f"Belge bulunuyor ve yükleniyor: {pdf_path}")

# PyPDFLoader ile belgeyi yüklüyoruz
loader = PyPDFLoader(pdf_path)
pages = loader.load()
print("Yükleme tamamlandı!")

Belge bulunuyor ve yükleniyor: data/CRE-10-2026-03-25_EN.pdf
Yükleme tamamlandı!


👉 `pages`'i keşfedin:
- Veri tipi nedir?
- Kaç sayfanız var?
- Bir sayfanın tipi nedir?
- Bir sayfanın içeriğine nasıl erişebilirsiniz?
- Tam belgenin kaç karakteri var?
- Bir sayfanın `metadata`'sında neler var?

In [7]:
# 1. Veri tipi nedir?
print(f"Veri tipi: {type(pages)}")

# 2. Kaç sayfanız var?
print(f"Toplam sayfa sayısı: {len(pages)}")

# 3. Bir sayfanın tipi nedir?
print(f"İlk sayfanın tipi: {type(pages[0])}")

# 4. Bir sayfanın içeriğine nasıl erişebilirsiniz? (İlk 150 karakteri görelim)
print(f"\nİlk sayfanın içeriği (örnek):\n{pages[0].page_content[:150]}...\n")

# 5. Tam belgenin kaç karakteri var?
toplam_karakter = sum(len(page.page_content) for page in pages)
print(f"Tüm belgenin toplam karakter sayısı: {toplam_karakter}")

# 6. Bir sayfanın metadata'sında neler var?
print(f"\nİlk sayfanın metadata'sı:\n{pages[0].metadata}")

Veri tipi: <class 'list'>
Toplam sayfa sayısı: 167
İlk sayfanın tipi: <class 'langchain_core.documents.base.Document'>

İlk sayfanın içeriği (örnek):
2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ...

Tüm belgenin toplam karakter sayısı: 512351

İlk sayfanın metadata'sı:
{'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '440386', 'dmxml.render.traceid': '69cbefa2424a75b9d296bcc090fe16ef', 'uid': 'eu.europa.europarl-DIN1-2026-0000086205_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-03-25_EN.pdf', 'total_pages': 167, 'page': 0, 'page_label': '1'}


## ✂️ Verilerimizi böl

Tam belgemiz gömülmek için çok uzun. Metin gömme aracımız 2.048 tokena kadar giriş alabilir. Gemini modelleri için bu yaklaşık 8.196 karakterdir (token başına 4 karakter).

Bu yüzden belgemizi daha küçük parçalara bölmek istiyoruz.

Zaten çalışabileceğimiz bir dizi sayfamız var. Ama sayfa sonları biraz keyfi: genellikle cümlenin ortasında görünürler.

Ayrıca, sayfalar arasında örtüşme yoktur. Bu yüzden bir sayfanın ilk satırı önceki tüm bağlamı kaçırır. Tam metni biraz örtüşmeyle bölmek daha iyidir.

İlk olarak, PDF'yi tekrar yükleyeceğiz, bu sefer bölmeden.

In [9]:
loader = PyPDFLoader(pdf_path, mode='single')
pdf = loader.load()
pdf_text = pdf[0].page_content
len(pdf_text)

512683

Artık tüm PDF'imizi tek bir belge olarak aldığımıza göre, onu daha akıllı bir şekilde parçalara bölebiliriz.

👉 Yine, ["Özyinelemeli olarak bölme" konusundaki LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter) gidin ve `pdf` _belgelerimizi_ parçalara (LangChain'de `documents` olarak adlandırılır) nasıl böleceğinizi öğrenin.

2_000 karakter (bizim durumumuzda yaklaşık yarım sayfa) parçalara 400 örtüşmeyle bölün. İsterseniz diğer değerlerle deneyebilirsiniz.

`RecursiveCharacterTextSplitter`'ın `.split_documents()` metodunu kullanın: bu metod giriş olarak bir belge alır ve bölünmüş belgeler çıktısı verir.

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Splitter ayarlıyoruz
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400
)

# PDF'imizi bu bıçakla parçalara ayırıyoruz
all_splits = text_splitter.split_documents(pdf)

👉 `all_splits`'i inceleyin:
- Veri tipi nedir?
- Kaç bölümünüz var?
- Bir bölümün tipi nedir?
- Bir bölümün içeriğine nasıl erişebilirsiniz?
- Şimdi toplamda kaç karakterimiz var?
- Bir bölümün `metadata`'sında neler var?

In [11]:
# 1. Veri tipi nedir?
print(f"Veri tipi: {type(all_splits)}")

# 2. Kaç bölümümüz var?
print(f"Toplam parça sayısı: {len(all_splits)}")

# 3. Bir bölümün tipi nedir?
print(f"İlk parçanın tipi: {type(all_splits[0])}")

# 4. Bir bölümün içeriğine nasıl erişebilirsiniz? (İlk 150 karakteri görelim)
print(f"\nİlk parçanın içeriği (örnek):\n{all_splits[0].page_content[:150]}...\n")

# 5. Şimdi toplamda kaç karakterimiz var? 
toplam_yeni_karakter = sum(len(split.page_content) for split in all_splits)
print(f"Parçaların toplam karakter sayısı: {toplam_yeni_karakter}")

# 6. Bir bölümün metadata'sında neler var?
print(f"\nİlk parçanın metadata'sı:\n{all_splits[0].metadata}")

Veri tipi: <class 'list'>
Toplam parça sayısı: 321
İlk parçanın tipi: <class 'langchain_core.documents.base.Document'>

İlk parçanın içeriği (örnek):
2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ...

Parçaların toplam karakter sayısı: 626200

İlk parçanın metadata'sı:
{'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '440386', 'dmxml.render.traceid': '69cbefa2424a75b9d296bcc090fe16ef', 'uid': 'eu.europa.europarl-DIN1-2026-0000086205_01.00-xm-01.00_text-xml', 'source': 'data/CRE-10-2026-03-25_EN.pdf', 'total_pages': 167}


## 🗄️ Her şeyi bir araya getir: belgelerimizi gömme ve vektör deposunda sakla

Elimizde şunlar var:
- Bir gömme aracı
- Veriyi yüklemek için bir yükleyici
- Belgemizi belgelere bölmek için bir metin bölücü

Neyi kaçırıyoruz?

Belgelerimizi gömebiliriz, ama onları bir yerde saklamak istiyoruz. İşte burada vektör deposu devreye girer: şunları saklamamıza olanak sağlar:
- belgeyi (parçayı),
- onun gömmesini,
- meta verilerini.

Sonraki adımda belgeleri verimli bir şekilde alabilecek olacağız.

👉 Bir `InMemoryVectorStore` nasıl oluşturabileceğinizi görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

In [13]:
from langchain_core.vectorstores import InMemoryVectorStore

# Vektör deposunu tertemiz baştan oluşturuyoruz
vectorstore = InMemoryVectorStore(embedding=embeddings)

# KOTAYI AŞMAMAK İÇİN: 167 sayfanın tamamını değil, sadece ilk 20 parçayı alıyoruz
kucuk_parcalar = all_splits[:20]

# Şimdi sadece bu küçük kısmı depoya atıyoruz
document_ids = vectorstore.add_documents(documents=kucuk_parcalar)

print(f"Harika! Kotaya takılmadan {len(kucuk_parcalar)} parça başarıyla depoya eklendi.")

Harika! Kotaya takılmadan 20 parça başarıyla depoya eklendi.


In [14]:
# Have a look at the first 3 document IDs
print(document_ids[:3])

['919fbfe4-2754-4852-93a2-2d657050c215', 'e67230d5-d671-45fd-b7b7-a86b55626b4a', 'a5f8eef5-9359-4d10-a830-3c90ffd90c5e']


In [15]:
# Use the vector store's `get_by_ids` method. You have to give it a list of document IDs.
cagrilan_belgeler = vectorstore.get_by_ids(document_ids[:3])

👉 Bir vektör deposundaki belgenin içeriğine ve meta verilerine nasıl erişebilirsiniz?

In [16]:
print("İlk çağrılan belgenin içeriği:\n")
print(cagrilan_belgeler[0].page_content)

İlk çağrılan belgenin içeriği:

2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI 
AUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN 
ISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD 
ΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES 
VERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR 
COMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV 
TUARASCÁIL FOCAL AR FHOCAL NA N-IMEACHTAÍ  DOBESEDNI ZAPISI RAZPRAV 
DOSLOVNO IZVJEŠĆE  SANATARKAT ISTUNTOSELOSTUKSET 
RESOCONTO INTEGRALE DELLE DISCUSSIONI  FULLSTÄNDIGT FÖRHANDLINGSREFERAT 
 
 
Сряда - Miércoles - Středa - Onsdag - Mittwoch - Kolmapäev - Τετάρτη - Wednesday 
Mercredi - Dé Céadaoin - Srijeda - Mercoledì - Trešdiena - Trečiadienis - Szerda - L-Erbgħa 
Woensdag - Środa - Quarta-feira - Miercuri - Stred

## 🔎 Benzer belgeleri almak için vektör deposunu kullan

Artık belgeleri gömleğe çevirdiğimize göre, benzer belgeleri almak için vektör deposunu kullanabiliriz.

👉 Bunun nasıl çalıştığını görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

Bir sorgu kullanın, örneğin "Tarım politikası üzerine tartışmayı özetle.", ve en benzer belgeleri bulun. Ayrıca alınacak belge sayısını da belirtebilirsiniz.

In [17]:
# Save your question into a variable called `query`
# Sorumuzu bir değişkene kaydediyoruz
query = "Summarize the debate on agricultural policy."

# Use the vector store to find similar documents to the query. Store the result in a variable called `retrieved_docs`
# Vektör deposunda benzerlik araması yapıyoruz ve en alakalı 3 parçayı getirmesini istiyoruz
retrieved_docs = vectorstore.similarity_search(query, k=3)

# Gelen en alakalı ilk parçanın içeriğine hızlıca bir göz atalım
print(f"Bulunan en alakalı parçanın ilk 200 karakteri:\n\n{retrieved_docs[0].page_content[:200]}...")


Bulunan en alakalı parçanın ilk 200 karakteri:

14 - European Citizens’ Initiative 'Ban on conversion practices in the European Union' 
(debate) ..........................................................................................................


Bu, RAG'ın sözde "Alma" (Retrieval) kısmını tamamlar: artık sorgumuza en benzer belgeleri bulabiliriz.

Çalışmanın çoğu artık tamamlandı!

## 💬 Sorumuza bir cevap üret

Şimdiye kadar benzer belgeleri almamızı sağlamak için sadece bir **gömme modeli** kullandık.

Şimdi, sorumuzla bir cevap almak için üretici bir LLM kullanacağız: ona aldığımız belgeler ve sorumuzla besleyeceğiz.

Bunu yapmanın en temel yolu tüm girdilerimizi birbirine bağlamak, sorumuzla eklemek ve sonucu görmek olacaktır.

Bir deneyelim.

👉 İlk olarak önceki meydan okumalarda olduğu gibi bir LLM başlatın.

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM'i başlatıyoruz (Yaratıcılığı sıfırlıyoruz ki sadece belgeden cevap versin)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

Sonra temel bir istem oluşturun:

In [19]:
prompt = '\n\n'.join([doc.page_content for doc in retrieved_docs])
prompt += "\n\n" + query

👉 Şimdi istemi kullanın:

In [21]:
from IPython.display import Markdown

# Hazırlanan prompt'u (bağlam + soru) modele gönderiyoruz
response = llm.invoke(prompt)

# Gelen cevabı ekrana basıyoruz
Markdown(response.content)

Based on the provided text, there is **no debate on agricultural policy**.

The listed items in the "SOMMAIRE" (Table of Contents) and "DE INHALT" (Content) are:

*   Statements by the President
*   Approval of minutes
*   Requests for waivers of immunity
*   Composition of political groups, committees, and delegations
*   Negotiations ahead of Parliament’s first reading
*   Order of business
*   Energy security, independence, and supply
*   Conclusions of the European Council meeting
*   European Citizens' Initiative 'Ban on conversion practices in the European Union'
*   Deposit protection and early intervention measures
*   Implementation of the Urban Wastewater Treatment Directive and risks to the security of supply of medicines
*   Imminent death penalty threats in Iran
*   Combating corruption
*   One-minute speeches on matters of political importance
*   Agenda of the next sitting
*   Closure of the sitting
*   Resumption and opening of the sitting

None of these topics directly relate to agricultural policy.

Bu fena değil, ama modele daha fazla rehberlik vererek daha kapsamlı bir istem yazarak daha iyisini yapabiliriz.

Bunu yapan ilk kişiler biz değilmişiz ve LangChain'in bizim için önceden hazırlanmış istem kütüphanesi var.

👉 Aşağıdaki hücreyi çalıştırın ve nasıl çalıştığını anlamaya çalışın. (LangSmithMissingAPIKeyWarning hakkında bir uyarı alacaksınız, bunu görmezden gelebilirsiniz.)

In [22]:
from langchain_classic import hub

prompt_template = hub.pull("rlm/rag-prompt")

example_messages = prompt_template.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

print("\n")
print(example_messages[0].content)



You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: (question goes here) 
Context: (context goes here) 
Answer:


LangChain'in bizim için nasıl daha kesin bir istem oluşturduğunu görüyor musunuz? Bunu RAG'ımız için kullanalım!

👉 İlk olarak, tüm alınan belgeleri iki yeni satırla ayrılmış tek bir uzun dizgiye birleştirin.

In [23]:
# İlk olarak, tüm alınan belgeleri iki yeni satırla ayrılmış tek bir uzun dizgiye birleştirin.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

formatted_context = format_docs(retrieved_docs)

# Ardından şablonu bir invoke nesnesine dönüştürmek için çağırıyoruz
# Az önce "hub"dan çektiğimiz profesyonel şablonu kullanıyoruz
messages = prompt_template.invoke({
    "context": formatted_context,
    "question": query
}).to_messages()

# Modeli çağırıyoruz
response = llm.invoke(messages)

# Sonucu yazdırıyoruz
Markdown(response.content)

This document does not contain information about a debate on agricultural policy. The provided text lists various other debates and agenda items, such as energy security, conversion practices, deposit protection, and corruption.

👉 Sonra, sorgunuz ve alınan belgelerden başlayarak bir `prompt` oluşturun. Yukarıdaki örneğe bakmayı unutmayın.

In [24]:
the_prompt = prompt_template.invoke({
    "context": formatted_context,
    "question": query
})

👉 Son olarak az önce oluşturduğumuz `the_prompt` ile LLM modelini kullanın:

In [25]:
response = llm.invoke(the_prompt)
Markdown(response.content)

I am sorry, but the provided context does not contain information about a debate on agricultural policy. Therefore, I cannot summarize it.

🎉 İlk RAG'ımızı tamamladık: LLM kendisine sağladığımız belgelerde ***temelli*** metin üretti.

## 💾 Gömmelerimizi kalıcı hale getir

Şimdiye kadar bellekte vektör deposuyla çalıştık. Bu yüzden not defterinizi kapattığınızda, tüm gömmeleri de kaybedeceksiniz.

⚠️ Bu gömmelerin sağlayıcınızın platformunda, bu durumda Google'ın makinelerinde çalışan modeller tarafından üretildiğini unutmayın. Ve bedava çalışmazlar. 💰

Bunun gibi bir, nispeten küçük belge için maliyet düşüktür, ama hızla artar. Şimdiye kadar sadece bir günün transkriptleriyle çalıştık. Oturum başına 3 tane daha, yılda 12 oturum, birden fazla yıl var...

Bunu çözmek için sadece vektör depomuzla kalıcı bir taneyi değiştireceğiz. Bu LangChain'in avantajıdır: bileşenleri değiştirmek çok kolay.

Bellekteki vektör depomuz deneme için harikaydı, şimdi başka bir taneyle değiştireceğiz. Çok popüler bir vektör deposu olan [Chroma](https://www.trychroma.com/)'yı kullanacağız. Bunu yerel olarak çalıştırabilir ve LangChain aracılığıyla kullanabiliriz.

Tüm akışımızı yeniden oluşturacağız. Her şeyi birkaç kod hücresinde tekrar bir araya getirmeye çalışmak iyi bir alıştırmadır. Aynı zamanda her şeyi yeniden kullanılabilir koda dönüştüreceğiz.

Sonunda iki fonksiyon istiyoruz:

1. `embed_and_store()`: Başka bir oturumun transkriptini vektör veritabanımıza ekle, böylece alacağımız daha fazla veri olsun.
2. `answer()`: Vektör depomuzla farklı sorularla sorgula.

#### 1. Bir Chroma vektör deposu başlat

👉 **Veri kalıcılığıyla** (yani verileri diskteki bir dizinde saklayarak) Chroma vektör deposunun nasıl oluşturulacağını görmek için [LangChain'in belgelerine](https://python.langchain.com/docs/integrations/vectorstores/chroma/) bakın.

In [26]:
from langchain_chroma import Chroma

# Verilerimizin kaydedileceği klasörün adını belirliyoruz
persist_directory = "./chroma_db"

# Chroma veritabanımızı başlatıyoruz
# Ona kullanacağı embedding modelini ve verileri nereye kaydedeceğini söylüyoruz
vector_store = Chroma(
    collection_name="eu_parliament_reports",
    embedding_function=embeddings,
    persist_directory=persist_directory
)

print(f"Chroma Vektör Deposu başlatıldı. Veriler '{persist_directory}' klasörüne kalıcı olarak kaydedilecek.")

Chroma Vektör Deposu başlatıldı. Veriler './chroma_db' klasörüne kalıcı olarak kaydedilecek.


#### 2. `embed_and_store()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [27]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def embed_and_store(file_path, vector_store):
    """Load a PDF file, split it into chunks, and store the chunks in a vector store."""
    
    # 1. Load the PDF file
    loader = PyPDFLoader(file_path)
    pages = loader.load()
    
    # 2. Split the pages into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
    all_splits = text_splitter.split_documents(pages)
    
    # 3. Add the chunks to the vector store
    # Ücretsiz API kotasına takılmamak için şimdilik sadece ilk 20 parçayı ekliyoruz. 
    # (Eğer ücretli/limitsiz bir API kullanıyor olsaydık all_splits'in tamamını verirdik)
    kucuk_parcalar = all_splits[:20]
    document_ids = vector_store.add_documents(documents=kucuk_parcalar)
    
    return document_ids

👉 Fonksiyonunuzu bir dosya veya hatta iki dosyayla deneyin:

In [28]:
# Fonksiyonumuzu elimizdeki PDF ile test ediyoruz
print("Dosya yükleniyor, parçalanıyor ve Chroma veritabanına ekleniyor...")

# pdf_path değişkenini daha önceki hücrelerde tanımlamıştık (data/CRE...pdf)
eklenen_idler = embed_and_store(pdf_path, vector_store)

print(f"İşlem tamam. Chroma veritabanına {len(eklenen_idler)} yeni parça kalıcı olarak eklendi.")

Dosya yükleniyor, parçalanıyor ve Chroma veritabanına ekleniyor...
İşlem tamam. Chroma veritabanına 20 yeni parça kalıcı olarak eklendi.


#### 3. `answer()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [31]:
def answer(query, vector_store, llm, prompt_template=None):
    """Answer a query using the vector store and the language model."""
    
    # 1. Benzer belgeleri çekiyoruz
    retrieved_docs = vector_store.similarity_search(query, k=6)
    
    # 2. Bağlamı (context) oluşturuyoruz
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    
    # 3. Şablon yoksa, 'langchain_classic' üzerinden çekiyoruz
    if not prompt_template:
        # Hata buradaydı: 'langchain' yerine 'langchain_classic' kullanıyoruz
        from langchain_classic import hub 
        prompt_template = hub.pull("rlm/rag-prompt")
    
    # İstemi (prompt) hazırlıyoruz
    prompt = prompt_template.invoke({
        "context": docs_content, 
        "question": query
    })
    
    # 4. Modeli çağırıyoruz
    response = llm.invoke(prompt)
    
    return response.content

👉 Fonksiyonunuzu beğendiğiniz bir sorguyla deneyin:

In [32]:
# Artık sistemimizi tek satırla test edebilirizz
soru = "What are the key points of the debate about corruption mentioned in the document?"

cevap = answer(soru, vector_store, llm)

from IPython.display import Markdown
Markdown(cevap)

The document mentions a debate on combating corruption. This topic is listed as item 18 in the agenda, with discussions on combating corruption taking place.

🏁 Tebrikler! Artık LangChain kullanarak RAG'da ustalaştınız ve vektör deponuza daha fazla belge eklemek ve onu sorgulamak için yeniden kullanılabilir fonksiyonlar yapmayı öğrendiniz.

## [İsteğe Bağlı] Meta veri ekleme

Kurduğumuz RAG, vektör deposundaki tüm belgeleri sorgular. Orada birden fazla yılın bilgisinin olduğunu düşünün. Yıllara veya tarihlere göre filtreyebilsek kullanışlı olurdu, değil mi?

Bunu nasıl yaparız? Vektör deposundaki belgelerin meta veri içerdiğini unutmayın. Eğer tarihi ekleyebilseydik, daha sonra filtrelemek için kullanabilirdik.

İpucu: Meta verilerinizi pipeline'ınızda olabildiğince erken ekleyin. Verileriniz vektör deposunda saklandıktan sonra eklemeye çalışmayın.

👉 `embed_and_store()` fonksiyonunuzu uyarlayın.

In [33]:
def embed_and_store_fancy(file_path, vector_store, session_date):
    """PDF yükle, parçala ve her parçaya oturum tarihini ekleyerek kaydet."""
    
    # 1. PDF Yükle
    loader = PyPDFLoader(file_path)
    pages = loader.load()
    
    # 2. Parçala
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
    all_splits = text_splitter.split_documents(pages)
    
    # 3. Her parçaya tarih bilgisini ekle 
    for split in all_splits:
        split.metadata['session_date'] = session_date
        
    # 4. Kotayı aşmamak için ilk 20 parçayı ekliyoruz
    kucuk_parcalar = all_splits[:20]
    document_ids = vector_store.add_documents(documents=kucuk_parcalar)
    
    return document_ids

👉 Fonksiyonunuzu deneyin ve vektör deponuzun ek meta veri içerdiğini kontrol edin.

In [34]:
# Belgemizin tarihi 25 Mart 2026
tarih = "2026-03-25"
fancy_ids = embed_and_store_fancy(pdf_path, vector_store, tarih)

print(f"Başarıyla {tarih} etiketiyle {len(fancy_ids)} parça eklendi.")

Başarıyla 2026-03-25 etiketiyle 20 parça eklendi.


Şimdi alıcıyı kullanıcının sorduğu tarihe göre sınırlamamız gerekiyor.

👉 `answer()` fonksiyonunuzu bir tarih alabilecek ve yeni meta verilere dayalı olarak belgeleri filtreleyebilecek şekilde uyarlayın.

In [35]:
def answer_with_filter(query, vector_store, llm, session_date):
    # Arama yaparken 'filter' parametresini ekliyoruz
    retrieved_docs = vector_store.similarity_search(
        query, 
        k=6, 
        filter={"session_date": session_date} # Sadece bu tarihe bak!
    )
    
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    
    from langchain_classic import hub
    prompt_template = hub.pull("rlm/rag-prompt")
    
    prompt = prompt_template.invoke({"context": docs_content, "question": query})
    response = llm.invoke(prompt)
    
    return response.content

In [36]:
soru = "What is mentioned about energy security?"
# Sadece 25 Mart 2026 tarihindeki tartışmalarda ara
sonuc = answer_with_filter(soru, vector_store, llm, session_date="2026-03-25")

Markdown(sonuc)

Energy security, independence, and supply in the geopolitical context are discussed, with a focus on ensuring market stability and affordable energy for industry and citizens. This topic is listed as item 11 in the contents of the document.

Harika! Güçlü bir RAG sistemi oluşturmak için benzerlik aramasını meta veri aramasıyla birleştirdiniz!